# Mutli Hop RAG

In [82]:
# !pip install langchain-chroma langchain-huggingface langsmith langchain-openai langchain-core

In [83]:
# !pip install --upgrade rank_bm25

In [84]:
# from rank_bm25 import BM25Okapi

In [85]:
from google.colab import drive
import os

# drive.mount('/content/drive')

In [86]:
# !pip -q install FlagEmbedding

In [87]:
import os
import shutil
import zipfile
import torch
from google.colab import drive, userdata
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# LLM 및 체인 구성을 위한 추가 라이브러리
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

# Pydantic Output Parser 구성 라이브러리
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

# API Key 설정
try:
    os.environ["LANGCHAIN_API_KEY"] = userdata.get('langgrpah')
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
    os.environ["LANGCHAIN_PROJECT"] = "multi-hop-rag"

    # OpenRouter 키 (변수명 확인 필요)
    openrouter_key = userdata.get('OPENROUTER')
except Exception as e:
    print(f"Key 설정 오류: {e}")

print("환경 설정 완료")

환경 설정 완료


In [88]:
from pydantic import BaseModel, Field
from typing import List, Optional

# 데이터 경로 설정
LOCAL_EXTRACT_PATH = "/content/drive/MyDrive/chroma_db_bge_m3 2"

# 데이터 셋업 함수
def setup_data():
    if os.path.exists(LOCAL_EXTRACT_PATH) and os.listdir(LOCAL_EXTRACT_PATH):
        print(f"로컬 데이터 경로 확인됨: {LOCAL_EXTRACT_PATH}")
        return True
    else:
        print(f"오류: 경로에 데이터가 없습니다 -> {LOCAL_EXTRACT_PATH}")
        return False


class Evidence(BaseModel):
    author: str = ""
    category: str = ""
    fact: str = Field(description="Exactly one single sentence copied verbatim from Context (Title line is allowed if present).")
    published_at: str = ""
    source: str = ""
    title: str = ""
    url: str = ""

class MultiHopResponse(BaseModel):
    reasoning: Optional[str] = ""  # 파싱 안정
    Answer: str = Field(description="Final answer. If question asks Who/Which individual/person, return the entity name. Otherwise return Yes/No/Insufficient information when applicable.")
    evidence_list: List[Evidence] = Field(default_factory=list)

class SelectedEvidenceResponse(BaseModel):
    selected_evidence: List[Evidence] = Field(default_factory=list)


if setup_data():
    print("데이터 및 스키마 준비 완료")

로컬 데이터 경로 확인됨: /content/drive/MyDrive/chroma_db_bge_m3 2
데이터 및 스키마 준비 완료


In [89]:
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
import re

def _tokenize(text: str):
    return re.findall(r"[A-Za-z0-9]+", (text or "").lower())

def _rrf_fuse(dense_docs, bm25_docs, k=60, w_dense=1.0, w_bm25=1.0):
    scores = {}

    def _key(d: Document):
        return (
            (d.metadata.get("url","") or "")
            + "||"
            + (d.metadata.get("original_title") or d.metadata.get("title") or "")
            + "||"
            + ((d.page_content[:80]) if d.page_content else "")
        )

    def _add(docs, w):
        for r, d in enumerate(docs):
            key = _key(d)
            scores[key] = scores.get(key, 0.0) + w * (1.0 / (k + r + 1))

    _add(dense_docs, w_dense)
    _add(bm25_docs, w_bm25)

    key2doc = {}
    for d in (dense_docs + bm25_docs):
        key2doc.setdefault(_key(d), d)

    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [key2doc[key] for key, _ in fused]

def _dedupe_cap_by_url(docs, cap_per_url=1):
    out, seen = [], {}
    for d in docs:
        url = d.metadata.get("url") or ""
        seen[url] = seen.get(url, 0) + 1
        if seen[url] <= cap_per_url:
            out.append(d)
    return out

def build_hybrid_retriever_from_chroma(vectorstore, dense_k=6, bm25_k=6, final_k=8, cap_per_url=1):
    # 여기 들어왔는지부터 찍기
    print("✅ build_hybrid_retriever_from_chroma: start")

    # 1) Dense retriever
    dense_retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": dense_k, "fetch_k": 40, "lambda_mult": 0.5}
    )

    # 2) Chroma -> docs
    store_data = vectorstore.get(include=["documents", "metadatas"])
    docs = [
        Document(page_content=txt, metadata=meta or {})
        for txt, meta in zip(store_data.get("documents", []), store_data.get("metadatas", []))
        if txt and isinstance(txt, str)
    ]
    print("✅ bm25 corpus size:", len(docs))
    if len(docs) == 0:
        raise ValueError("❌ BM25 corpus가 0임. vectorstore.get() 결과 확인 필요")

    tokenized_corpus = [_tokenize(d.page_content) for d in docs]
    bm25_index = BM25Okapi(tokenized_corpus)

    def bm25_retrieve(q: str, k: int):
        q_tokens = _tokenize(q)
        scores = bm25_index.get_scores(q_tokens)
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [docs[i] for i in top_idx]

    # 3) Hybrid
    def _hybrid(q: str):
        # ✅ 최신 LCEL 방식: invoke
        dense_docs = dense_retriever.invoke(q)[:dense_k]
        bm25_docs  = bm25_retrieve(q, bm25_k)
        fused = _rrf_fuse(dense_docs, bm25_docs, k=60, w_dense=1.0, w_bm25=1.0)
        fused = _dedupe_cap_by_url(fused, cap_per_url=cap_per_url)
        return fused[:final_k]

    out = RunnableLambda(_hybrid)

    print("✅ build_hybrid_retriever_from_chroma: return", type(out))
    return out

In [90]:
SYSTEM_GLOBAL = r"""
You are an evidence-grounded multi-hop QA assistant. Use ONLY the provided Context.

RULES (must follow):
1) Evidence-first: select 2~6 evidence items BEFORE answering.
   - evidence.fact MUST be EXACTLY one single sentence copied verbatim from Context.
   - Title lines are allowed ONLY if they appear in Context (e.g., "Title: ...").

2) Answer using ONLY selected_evidence.

question_type:
- Use the provided question to decide question_type. Do NOT infer question_type from missing evidence.
- temporal_query: any explicit date/year/month OR ordering words (before/after/when/first/last).
- comparison_query: two distinct entities/claims AND compare language (vs/compare/same/different).
- inference_query: cross-source linking to identify the same entity/claim without explicit time/compare.
- null_query: selected_evidence is empty or not answer-bearing.

EVIDENCE DIVERSITY (HARD):
- selected_evidence MUST contain at most ONE item per unique url. (One evidence per document.)
- For inference_query: if you can find answer-bearing evidence from at least TWO different urls in Context, select them.
- If you cannot find TWO answer-bearing urls, return Insufficient information.
- If the question explicitly names sources (e.g., "Fortune and TechCrunch"), you MUST include at least one evidence from EACH named source if present; otherwise Insufficient information.

OUTPUT (JSON ONLY; no extra text):
{{
  "question_type": "comparison_query|temporal_query|inference_query|null_query",
  "selected_evidence": [ {{"author":"","category":"","fact":"","published_at":"","source":"","title":"","url":""}} ],
  "Answer": "",
  "evidence_list": [ {{"author":"","category":"","fact":"","published_at":"","source":"","title":"","url":""}} ]
}}

Output Rules (Strict):
- For WHO (person/individual) questions, output ONLY the person's full name.
- Do NOT include explanations, citations, or extra words in the Answer.
- For other questions, keep the Answer as short as possible.
- If the answer cannot be determined, output exactly: Insufficient information


Output constraint:
- evidence_list should be the same as selected_evidence (same items, same order). If you already filled selected_evidence, copy it into evidence_list.
- If insufficient, you MUST still output the full JSON schema with:
  "question_type":"null_query", "selected_evidence":[], "Answer":"Insufficient information", "evidence_list":[]
"""


In [91]:
# Retrieval
def get_global_retriever(persist_dir):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Embedding Model on {device}...")

    embedding_model = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )

    # chroma.sqlite3 실제 위치 찾기
    real_db_path = persist_dir
    for root, dirs, files in os.walk(persist_dir):
        if "chroma.sqlite3" in files:
            real_db_path = root
            break

    vectorstore = Chroma(
        persist_directory=real_db_path,
        embedding_function=embedding_model,
        collection_name="multihop_rag"
    )

    # 컬렉션 카운트 확인
    cnt = vectorstore._collection.count()
    print("collection count:", cnt)
    if cnt == 0:
        raise ValueError("collection count가 0. DB 경로/collection_name 확인 필요")

    # 여기서부터: 어떤 에러든 숨기지 말고 무조건 출력 + raise
    try:
        hybrid_retriever = build_hybrid_retriever_from_chroma(
            vectorstore,
            dense_k=6,
            bm25_k=6,
            final_k=8,
            cap_per_url=1
        )
    except Exception as e:
        print("❌ build_hybrid_retriever_from_chroma 실패:", repr(e))
        raise

    # 반환값 검증
    if hybrid_retriever is None:
        raise ValueError("❌ hybrid_retriever가 None으로 반환됨 (build_hybrid_retriever_from_chroma 확인)")

    # invoke가 되는 객체인지 체크 (Runnable/리트리버 형태)
    if not hasattr(hybrid_retriever, "invoke"):
        raise TypeError(f"❌ hybrid_retriever에 invoke가 없음: type={type(hybrid_retriever)}")

    print("✅ hybrid retriever ready:", type(hybrid_retriever))
    return hybrid_retriever

In [92]:
import json
import re
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser
from typing import List, Dict, Any
from FlagEmbedding import FlagReranker


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_GLOBAL),
    ("human",
     "Context:\n{context}\n\n"
     "Question:\n{question}")
])

TYPE_SYSTEM = """You are a strict classifier.
Classify the question into EXACTLY one label:
- temporal_query
- comparison_query
- inference_query

Return JSON only: {{"question_type": "<label>"}}.
No extra keys, no explanation.
"""

type_prompt = ChatPromptTemplate.from_messages([
    ("system", TYPE_SYSTEM),
    ("human", "{question}")
])

def safe_parse_type(text: str) -> str:
    t = (text or "").strip()
    m = re.search(r"(\{.*\})", t, flags=re.DOTALL)
    if m:
        t = m.group(1)
    try:
        obj = json.loads(t)
        qt = obj.get("question_type")
        if qt in ["temporal_query", "comparison_query", "inference_query"]:
            return qt
    except:
        pass
    return "inference_query"

def classify_question_type_llm(llm, question: str) -> str:
    msg = type_prompt.invoke({"question": question})
    r = llm.invoke(msg)
    return safe_parse_type(r.content if hasattr(r, "content") else str(r))



In [93]:
TEMPORAL_OVERRIDE_PAT = re.compile(
    r"\bpublished\b|\blater\b|\bearlier\b|\bbefore\b|\bafter\b|\btimeline\b|\bwhen\b|\bdate\b|\byear\b|"
    r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b|"
    r"\b(19|20)\d{2}\b",
    re.I
)

def enforce_dataset_style_qtype(question: str, qtype_llm: str) -> str:
    if TEMPORAL_OVERRIDE_PAT.search(question):
        return "temporal_query"
    return qtype_llm


def temporal_bonus(c):
    s = (c.get("sent") or "")
    return 0.3 if TEMPORAL_OVERRIDE_PAT.search(s) else 0.0

def pick_balanced_by_source(cands_sorted, top_n=24, per_source=6):
    buckets = {}
    for c in cands_sorted:
        src = c.get("source") or ""
        buckets.setdefault(src, []).append(c)

    picked = []
    for src, lst in buckets.items():
        picked.extend(lst[:per_source])

    picked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return picked[:top_n]


In [94]:
import re
from typing import List, Dict, Any
from FlagEmbedding import FlagReranker

# -----------------------------
# Sentence split + WHO / PERSON patterns
# -----------------------------
_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+|\n+')

# "who/which individual/person" + 약간 넓게
WHO_Q_PAT = re.compile(
    r"^\s*(who\b|which\s+(individual|person)\b)",
    re.I
)

# 사람 이름처럼 보이는 패턴 (2~4 토큰)
# - 이니셜: "J. K. Rowling"
# - 하이픈: "Sam Bankman-Fried"
# - 아포스트로피: "O'Connor" 일부 커버
PERSON_NAME_PAT = re.compile(
    r"\b(?:[A-Z]\.\s*){0,3}[A-Z][a-z]+(?:[-'][A-Za-z]+)?(?:\s+[A-Z][a-z]+(?:[-'][A-Za-z]+)?){1,3}\b"
)

def is_who_question(question: str) -> bool:
    return bool(WHO_Q_PAT.search(question or ""))

def who_bonus(question: str, c: Dict[str, Any], qtype: str) -> float:
    """
    Soft bonus:
    - ONLY when (qtype == inference_query) AND question is WHO-type
    - sentence looks like contains a person name
    - avoid boosting titles (optional but recommended)
    """
    if qtype != "inference_query":
        return 0.0
    if not is_who_question(question):
        return 0.0

    # title 후보는 과하게 튀는 경우가 많아서 기본적으로 제외
    if c.get("is_title"):
        return 0.0

    s = (c.get("sent") or "").strip()
    if len(s) < 15:
        return 0.0

    return 0.25 if PERSON_NAME_PAT.search(s) else 0.0


# -----------------------------
# Doc text extractor
# -----------------------------
def _get_text_and_md(d):
    # LangChain Document 케이스
    if hasattr(d, "page_content"):
        text = (getattr(d, "page_content", "") or "")
        md = getattr(d, "metadata", {}) or {}
        return text, md

    # dict 케이스
    if isinstance(d, dict):
        md = d.get("metadata", {}) or {}
        text = d.get("page_content") or d.get("text") or d.get("content") or ""
        return text, md

    return "", {}


# -----------------------------
# Sentence candidates builder
# -----------------------------
def docs_to_sentence_candidates(docs, max_sent_per_doc: int = 25) -> List[Dict[str, Any]]:
    cands = []
    for d in docs or []:
        text, md = _get_text_and_md(d)

        title = (md.get("original_title") or md.get("title") or "").strip()
        url = (md.get("url") or "").strip()
        source = (md.get("source") or "").strip()
        published_at = str(md.get("published_at") or "").strip()

        text = (text or "").strip()

        # text가 비어도 title은 후보로 넣을지 선택
        if not text:
            if title:
                cands.append({
                    "sent": title,
                    "title": title,
                    "url": url,
                    "source": source,
                    "published_at": published_at,
                    "is_title": True
                })
            continue

        # 문장 분리
        sents = [s.strip() for s in _SENT_SPLIT.split(text) if s and s.strip()]
        if not sents:
            sents = [text]

        sents = sents[:max_sent_per_doc]

        # title도 후보로 넣되, is_title 표시
        if title:
            cands.append({
                "sent": title,
                "title": title,
                "url": url,
                "source": source,
                "published_at": published_at,
                "is_title": True
            })

        for s in sents:
            if len(s) < 10:
                continue
            cands.append({
                "sent": s,
                "title": title,
                "url": url,
                "source": source,
                "published_at": published_at,
                "is_title": False
            })

    return cands


# -----------------------------
# Reranker (bge-reranker-large)
# -----------------------------
RERANKER = FlagReranker("BAAI/bge-reranker-large", use_fp16=True)

def rerank_sentences_typed(
    question: str,
    cands: List[Dict[str, Any]],
    qtype: str,
    top_n: int = 24,
    per_url_cap: int = 5,
    per_source: int = 6
):
    if not cands:
        return []

    pairs = [(question, c["sent"]) for c in cands]
    scores = RERANKER.compute_score(pairs)

    for c, s in zip(cands, scores):
        base = float(s)

        # temporal 보너스는 temporal_query에서만
        if qtype == "temporal_query":
            base += temporal_bonus(c)

        # who 보너스는 inference_query + who 질문에서만
        base += who_bonus(question, c, qtype)

        c["rerank_score"] = base

    cands_sorted = sorted(cands, key=lambda x: x["rerank_score"], reverse=True)

    # comparison이면 source 균형 우선
    if qtype == "comparison_query":
        cands_sorted = pick_balanced_by_source(
            cands_sorted, top_n=top_n, per_source=per_source
        )

    # URL 다양성은 마지막에만
    out, url_cnt = [], {}
    for c in cands_sorted:
        url = c.get("url") or ""
        url_cnt[url] = url_cnt.get(url, 0) + 1
        if url and url_cnt[url] > per_url_cap:
            continue
        out.append(c)
        if len(out) >= top_n:
            break

    return out


# -----------------------------
# Context formatter
# -----------------------------
def format_sentence_context(top_sents: List[Dict[str, Any]]) -> str:
    lines = []
    for i, c in enumerate(top_sents, 1):
        title = c.get("title", "")
        url = c.get("url", "")
        source = c.get("source", "")
        published_at = c.get("published_at", "")
        sent = c.get("sent", "")
        is_title = c.get("is_title", False)

        lines.append(
            f"[{i}] sentence: {sent}\n"
            f"    Title: {title}\n"
            f"    Source: {source}\n"
            f"    Published_at: {published_at}\n"
            f"    URL: {url}\n"
            f"    is_title: {is_title}"
        )
    return "\n\n".join(lines)


In [95]:
def create_chain():
    retriever = get_global_retriever(LOCAL_EXTRACT_PATH)

    llm = ChatOpenAI(
        model="openai/gpt-4o-mini",
        openai_api_key=userdata.get('OPENROUTER'),
        openai_api_base="https://openrouter.ai/api/v1",
        temperature=0,
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com",
            "X-Title": "MULTI HOP RAG Project"
        }
    )

    def assert_str(x):
        # 디버그용: dict 섞이는 순간 바로 잡기
        if not isinstance(x, str):
            print(">>> NOT STR! type:", type(x), "preview:", str(x)[:200])
            raise TypeError(f"Expected str but got {type(x)}")
        return x

    def _answer(question: str, context: str):
        messages = prompt.invoke({"context": context, "question": question, "hint": ""})
        return llm.invoke(messages)

    chain = (
        RunnablePassthrough()
        | RunnableLambda(assert_str)  # 입력은 무조건 str
        | RunnableLambda(lambda q: {"question": q, "qtype": enforce_dataset_style_qtype(q, classify_question_type_llm(llm, q))})
        | RunnableLambda(lambda x: {**x, "docs": retriever.invoke(assert_str(x["question"]))})
        | RunnableLambda(lambda x: {**x, "cands": docs_to_sentence_candidates(x["docs"], max_sent_per_doc=200)})
        | RunnableLambda(lambda x: {**x, "top_sents": rerank_sentences_typed(
            x["question"], x["cands"], x["qtype"], top_n=24, per_url_cap=5, per_source=6
        )})
        | RunnableLambda(lambda x: {**x, "context": format_sentence_context(x["top_sents"])})
        | RunnableLambda(lambda x: {**x, "llm_out": _answer(x["question"], x["context"])})
        | RunnableLambda(lambda x: {**x, "parsed": safe_parse_output(
            x["llm_out"].content if hasattr(x["llm_out"], "content") else str(x["llm_out"])
        )})
        | RunnableLambda(lambda x: {**x, "parsed": force_fill_evidence(
            x["parsed"], x["top_sents"], k=3 if x["qtype"] == "temporal_query" else 2
        )})
        | RunnableLambda(lambda x: {**x, "parsed": {**x["parsed"], "question_type": (
            "null_query" if (x["parsed"].get("Answer","").strip() == "Insufficient information") else x["qtype"]
        )}})
        | RunnableLambda(lambda x: normalize_schema(x["parsed"]))

    )

    return chain


In [96]:
def safe_parse_output(text: str):
    t = (text or "").strip()

    # 모델이 가끔 JSON 대신 이것만 던짐
    if t.lower() in ["insufficient information", "insufficient", "not enough information"]:
        return {
            "question_type": "null_query",
            "selected_evidence": [],
            "Answer": "Insufficient information",
            "evidence_list": []
        }

    # ```json ... ``` 또는 ``` ... ``` 제거 (안전하게)
    m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", t, flags=re.DOTALL)
    if m:
        t = m.group(1).strip()

    # JSON 파싱
    try:
        return json.loads(t)
    except Exception:
        # 파싱 실패시에도 죽지 말고 fallback (디버깅용 raw 포함)
        return {
            "question_type": "null_query",
            "selected_evidence": [],
            "Answer": "Insufficient information",
            "evidence_list": [],
            "_raw": (text or "")[:2000]  # 너무 길면 컷
        }

def force_fill_evidence(out: dict, top_sents: List[Dict[str, Any]], k: int = 2) -> dict:
    if not isinstance(out, dict):
        return out

    # ✅ Insufficient이면 근거도 비워서 혼동 방지 (이건 너가 한 게 정답)
    if (out.get("Answer") or "").strip() == "Insufficient information":
        out["selected_evidence"] = []
        out["evidence_list"] = []
        return out

    # 이미 evidence가 있으면 그대로 유지
    evs = out.get("evidence_list") or []
    sel = out.get("selected_evidence") or []
    if len(evs) >= 1 or len(sel) >= 1:
        return out

    filled = [
        {
            "title": s.get("title",""),
            "author": "",
            "url": s.get("url",""),
            "source": s.get("source",""),
            "category": "",
            "published_at": s.get("published_at",""),
            "fact": s.get("sent","")
        }
        for s in (top_sents or [])[:k]
    ]

    # ✅ 둘 다 채워서 스키마 정합성 확보
    out["selected_evidence"] = filled
    out["evidence_list"] = filled
    return out


def normalize_schema(out: dict) -> dict:
    # 1) 타입 방어
    if not isinstance(out, dict):
        return {
            "question_type": "null_query",
            "selected_evidence": [],
            "Answer": "Insufficient information",
            "evidence_list": []
        }

    # 2) Answer 키 통일 (answer -> Answer)
    if "Answer" not in out and "answer" in out:
        out["Answer"] = out.get("answer")
    out.pop("answer", None)

    # 3) 필수 키 기본값
    out.setdefault("question_type", "null_query")
    out.setdefault("selected_evidence", [])
    out.setdefault("evidence_list", [])

    # 4) evidence 동기화 (네 SYSTEM_GLOBAL의 하드룰 반영)
    sel = out.get("selected_evidence") or []
    evs = out.get("evidence_list") or []

    if sel and not evs:
        out["evidence_list"] = sel
    elif evs and not sel:
        out["selected_evidence"] = evs
    else:
        # 둘 다 있으면 selected_evidence를 기준으로 통일 (한쪽만 믿기)
        out["evidence_list"] = sel

    # 5) Answer 비었으면 처리
    ans = (out.get("Answer") or "").strip()
    if not ans:
        out["Answer"] = "Insufficient information"
        out["question_type"] = "null_query"
        out["selected_evidence"] = []
        out["evidence_list"] = []

    return out


def minimal_guardrail(out: dict) -> dict:
    if not isinstance(out, dict):
        return {
            "Answer": "Insufficient information",
            "evidence_list": []
        }

    ans = (out.get("Answer") or "").strip()
    if not ans:
        out["Answer"] = "Insufficient information"

    if "evidence_list" not in out or out["evidence_list"] is None:
        out["evidence_list"] = []

    return out

In [97]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from datasets import load_dataset

# QA 데이터셋 로드
try:
    qa_df
except NameError:
    print("QA 데이터셋 로드 중...")
    ds = load_dataset("yixuantt/MultiHopRAG", "MultiHopRAG", split="train")
    qa_df = ds.to_pandas()

# 층화 추출 (Stratified Sampling) - 비율 맞춰서 50개 뽑기
print(f"전체 데이터 개수: {len(qa_df)}")
print("Question Type별 비율에 맞춰 50개 샘플링 중...")

# sklearn을 사용하여 비율 유지하며 추출
sampled_df, _ = train_test_split(
    qa_df,
    train_size=50,
    stratify=qa_df['question_type'],
    random_state=42 # 재현성을 위해 시드 고정
)

print(f"추출된 샘플 개수: {len(sampled_df)}")
print(sampled_df['question_type'].value_counts()) # 타입별 개수 확인

전체 데이터 개수: 2556
Question Type별 비율에 맞춰 50개 샘플링 중...
추출된 샘플 개수: 50
question_type
comparison_query    17
inference_query     16
temporal_query      11
null_query           6
Name: count, dtype: int64


In [98]:
# ==============================================================================
# 2. 체인 생성 (초기화)
# ==============================================================================
print("2. RAG 체인 초기화 중...")

# [중요] create_chain()이 정의되어 있는지 확인하고 실행
if 'create_chain' not in globals():
    raise NameError("❌ 'create_chain' 함수가 정의되지 않았습니다! 위쪽 셀에서 create_chain 및 관련 함수들을 먼저 실행해주세요.")

try:
    # 체인 객체 생성 (변수명 chain에 할당)
    chain = create_chain()
    print("✅ 체인 생성 완료!")
except Exception as e:
    print(f"❌ 체인 생성 중 오류 발생: {e}")
    print("Helper 함수(retriever 등)가 정의되었는지 확인해주세요.")
    raise e # 오류 발생 시 여기서 멈춤

# ==============================================================================
# 3. RAG 추론 실행
# ==============================================================================
print("3. 추론 시작 (Inference Loop)...")

rag_answers = []
rag_evidence_lists = []

for index, row in tqdm(sampled_df.iterrows(), total=len(sampled_df), desc="Inference"):
    query_text = row['query']

    try:
        # 체인 호출
        result = chain.invoke(query_text)

        # 결과 파싱 (normalize_schema의 리턴값인 dict 구조 가정)
        # 키 이름 대소문자 등 다양한 경우의 수 처리
        ans = result.get('answer') or result.get('Answer') or "No Answer"
        ev_list = result.get('evidence_list') or result.get('evidence') or []

        rag_answers.append(ans)
        rag_evidence_lists.append(ev_list)

    except Exception as e:
        # 오류 발생 시 로그 출력 후 'Error' 처리 (멈추지 않음)
        # print(f" [Error] Index {index}: {e}")
        rag_answers.append("Error")
        rag_evidence_lists.append([])

# ==============================================================================
# 4. 결과 DataFrame 구성
# ==============================================================================
final_eval_df = sampled_df.copy()

# RAG 결과 컬럼 추가
final_eval_df['RAG_answer'] = rag_answers
final_eval_df['RAG_evidence_list'] = rag_evidence_lists

# 요청하신 컬럼 순서대로 정리
target_columns = ["query", "evidence_list", "RAG_evidence_list", "answer", "RAG_answer"]
final_eval_df = final_eval_df[target_columns]

print("\n=== 생성 완료 ===")
print("상위 3개 결과 미리보기:")
display(final_eval_df.head(3))

# CSV 저장 (필요 시 주석 해제)
final_eval_df.to_csv("rag_chain_evaluation_results.csv_ej", index=False, encoding="utf-8-sig")

2. RAG 체인 초기화 중...
Loading Embedding Model on cuda...
collection count: 2157
✅ build_hybrid_retriever_from_chroma: start
✅ bm25 corpus size: 2157
✅ build_hybrid_retriever_from_chroma: return <class 'langchain_core.runnables.base.RunnableLambda'>
✅ hybrid retriever ready: <class 'langchain_core.runnables.base.RunnableLambda'>
✅ 체인 생성 완료!
3. 추론 시작 (Inference Loop)...


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 91.93it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 84.46it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 93.39it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 85.81it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 108.92it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 121.33it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 76.18it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 90.43it/s]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 104.15it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 85.64it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 94.94it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 107.62it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 120.53it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 102.10it/s]

pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 100.90it/s]

pre tokenize: 100%|██████████| 3/3 [00:00<00:00, 87.30it/s]

pre tokenize: 100


=== 생성 완료 ===
상위 3개 결과 미리보기:


,query,evidence_list,RAG_evidence_list,answer,RAG_answer
2084,"Does the TechCrunch article suggest that ""Peop...","[{'author': 'Sarah Perez', 'category': 'techno...","[{'author': 'TechCrunch', 'category': '', 'fac...",Yes,Yes
142,Who is the individual targeted by Attorney Gen...,"[{'author': 'Michael R. Sisak, The Associated ...","[{'author': 'Fortune', 'category': '', 'fact':...",Donald Trump,Donald Trump
2441,"Between the report by The Age on October 22, 2...","[{'author': 'Kyle Wiggers', 'category': 'techn...","[{'author': '', 'category': '', 'fact': 'She a...",Yes,Yes


In [99]:
# @title 평가지표
import pandas as pd
import ast
import re
import string
from collections import Counter
# import os
# from google.colab import drive, userdata


# # 1. 구글 드라이브 마운트 (데이터가 드라이브에 있으므로 필수)
# if not os.path.exists('/content/drive'):
#     drive.mount('/content/drive')

# 1. 파일 로드 (경로는 사용자 환경에 맞게 유지)
df = pd.read_csv("rag_chain_evaluation_results.csv_ej", encoding="utf-8-sig")

# ------------------------------------------------------------------
# [Helper] 파싱 함수
# ------------------------------------------------------------------
def parse_list_robust(x):
    if not isinstance(x, str): return []
    try:
        return ast.literal_eval(x)
    except:
        pass
    fixed_str = re.sub(r'\}\s*\{', '}, {', x)
    try:
        return ast.literal_eval(fixed_str)
    except:
        return []

df['evidence_list'] = df['evidence_list'].apply(parse_list_robust)
df['RAG_evidence_list'] = df['RAG_evidence_list'].apply(parse_list_robust)

# ------------------------------------------------------------------
# [Metric 1] Retrieval Metrics (Hit, MRR, MAP)
# ------------------------------------------------------------------
def calculate_retrieval_metrics(row, k=5):
    gold_urls = set([item.get('url') for item in row['evidence_list'] if item.get('url')])
    raw_retrieved_urls = [item.get('url') for item in row['RAG_evidence_list'] if item.get('url')]

    # 중복 제거 (순서 유지)
    retrieved_urls = []
    seen = set()
    for url in raw_retrieved_urls:
        if url not in seen:
            retrieved_urls.append(url)
            seen.add(url)
    retrieved_urls = retrieved_urls[:k]

    # Hit@K
    hit = 1 if not gold_urls.isdisjoint(retrieved_urls) else 0

    # MRR@K
    mrr = 0
    for i, url in enumerate(retrieved_urls):
        if url in gold_urls:
            mrr = 1 / (i + 1)
            break

    # MAP@K
    num_gold = len(gold_urls)
    if num_gold == 0:
        ap = 0
    else:
        hits = 0
        sum_precisions = 0
        for i, url in enumerate(retrieved_urls):
            if url in gold_urls:
                hits += 1
                sum_precisions += hits / (i + 1)
        ap = sum_precisions / num_gold

    return pd.Series([hit, mrr, ap], index=[f'Hit@{k}', f'MRR@{k}', f'MAP@{k}'])

# [Metric 2] Answer Match Accuracy (Exact Match & F1)
def normalize_answer(s):
    """
    평가를 위해 답변 텍스트를 정규화합니다.
    (소문자 변환, 문장부호 제거)
    """

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)

    def lower(text):
        return str(text).lower()

    if not s or pd.isna(s): return ""

    return white_space_fix(remove_punc(lower(s)))

def calculate_qa_metrics(row):
    """
    정답(Answer)과 모델 예측(RAG_Answer)을 비교합니다.
    """
    gold_text = normalize_answer(row['answer'])
    pred_text = normalize_answer(row['RAG_answer'])

    # Exact Match (EM): 완전히 일치하는가? (0 or 1)
    em = 1 if gold_text == pred_text else 0


    return pd.Series([em], index=['Exact Match Accuracy'])

# ------------------------------------------------------------------
# [Execution] 전체 적용 및 저장
# ------------------------------------------------------------------
K_VALUE = 5

# 1. Retrieval Score 계산
retrieval_scores = df.apply(lambda row: calculate_retrieval_metrics(row, k=K_VALUE), axis=1)

# 2. QA Score 계산 (추가된 부분)
qa_scores = df.apply(calculate_qa_metrics, axis=1)

# 3. 전체 데이터프레임 병합
final_df = pd.concat([df, retrieval_scores, qa_scores], axis=1)

# 결과 출력 (평균 점수 확인)
print(f"======== Evaluation Results (K={K_VALUE}) ========")
# 백분율(%)로 변환하여 출력
summary = final_df[[f'Hit@{K_VALUE}', f'MRR@{K_VALUE}', f'MAP@{K_VALUE}', 'Exact Match Accuracy']].mean() * 100
print(summary)

# CSV 저장
save_path = "/content/drive/MyDrive/rag_evaluation_with_all_metrics_ej.csv"
final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
print(f"\n[Done] 결과 파일이 저장되었습니다: {save_path}")

======== Evaluation Results (K=5) ========
Hit@5                   74.000000
MRR@5                   70.000000
MAP@5                   48.083333
Exact Match Accuracy    70.000000
dtype: float64

[Done] 결과 파일이 저장되었습니다: /content/drive/MyDrive/rag_evaluation_with_all_metrics_ej.csv


In [100]:
chain = create_chain()
q = "Considering the economic analysis from Bloomberg and the agricultural developments reported by Reuters, which minister, responsible for the finance portfolio in Zimbabwe, also announced a partnership with an international firm to boost crop production in the country?"
print(chain.invoke(q))

Loading Embedding Model on cuda...
collection count: 2157
✅ build_hybrid_retriever_from_chroma: start
✅ bm25 corpus size: 2157
✅ build_hybrid_retriever_from_chroma: return <class 'langchain_core.runnables.base.RunnableLambda'>
✅ hybrid retriever ready: <class 'langchain_core.runnables.base.RunnableLambda'>


Compute Scores: 100%|██████████| 2/2 [00:00<00:00,  7.75it/s]


{'question_type': 'null_query', 'selected_evidence': [], 'Answer': 'Insufficient information', 'evidence_list': []}
